In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, ToggleButtons, HBox, VBox, HTML, Layout, interactive
from IPython.display import display


# ============================================================
# ZERO-INPUT LIMIT CYCLES IN A FIRST-ORDER IIR FILTER
# ============================================================


# ------------------------------------------------------------
# Quantizer
# ------------------------------------------------------------

def quantize(x, K, mode):

    Delta = 2.0**(-K)

    if mode == 'Rounding':

        q = Delta * np.round(np.asarray(x) / Delta)

    else:

        q = Delta * np.trunc(np.asarray(x) / Delta)


    q = np.clip(q, -1.0, 1.0 - Delta)

    q = Delta * np.round(q / Delta)


    if np.ndim(q) == 0:

        return float(q)

    return q


# ------------------------------------------------------------
# First-order zero-input simulation
# ------------------------------------------------------------

def simulate_first_order(alpha, K, y0, samples, mode):

    y_quantized = np.zeros(samples)

    y_ideal = np.zeros(samples)


    y_quantized[0] = quantize(y0, K, mode)

    y_ideal[0] = y_quantized[0]


    for n in range(1, samples):

        y_ideal[n] = alpha * y_ideal[n - 1]

        y_quantized[n] = quantize(alpha * y_quantized[n - 1], K, mode)


    return y_ideal, y_quantized


# ------------------------------------------------------------
# Limit-cycle detector
# ------------------------------------------------------------

def detect_limit_cycle(y, K, max_period=12, repetitions=3):

    Delta = 2.0**(-K)

    q = np.round(np.asarray(y) / Delta).astype(int)


    for period in range(1, max_period + 1):

        required = period * repetitions


        if len(q) < required:

            continue


        tail = q[-required:]

        reference = tail[-period:]


        cycle_found = True


        for r in range(2, repetitions + 1):

            segment = tail[-r * period:-(r - 1) * period]


            if not np.array_equal(reference, segment):

                cycle_found = False

                break


        if cycle_found:

            return period, reference.astype(float) * Delta


    return None, None


# ------------------------------------------------------------
# CSS
# ------------------------------------------------------------

style_html = HTML("""
<style>

.lc-root {
    width: 960px;
    max-width: 960px;
    font-family: Arial, sans-serif;
}

.lc-title {
    background: #26384a;
    color: white;
    padding: 10px 14px;
    border-radius: 7px;
    font-size: 20px;
    font-weight: bold;
    margin-bottom: 8px;
}

.lc-card-row {
    display: flex;
    gap: 8px;
    margin-bottom: 8px;
}

.lc-card {
    flex: 1;
    background: #f6f8fa;
    border: 1px solid #d3d9df;
    border-radius: 7px;
    padding: 8px 10px;
    font-size: 12.5px;
    line-height: 1.35;
    box-sizing: border-box;
}

.lc-card-title {
    font-weight: bold;
    margin-bottom: 4px;
    color: #26384a;
}

.lc-controls {
    border: 1px solid #d3d9df;
    border-radius: 7px;
    padding: 7px 9px;
    background: #fbfcfd;
    margin-bottom: 6px;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea {
    overflow-x: visible !important;
    max-width: none !important;
}

</style>
""")


# ------------------------------------------------------------
# Header
# ------------------------------------------------------------

title_html = HTML("""
<div class="lc-root">

    <div class="lc-title">
        Zero-Input Limit Cycles in a First-Order IIR Filter
    </div>

    <div class="lc-card-row">

        <div class="lc-card">

            <div class="lc-card-title">
                How it works
            </div>

            The zero-input recursion
            <b>y[n] = Q(αy[n−1])</b>
            is evaluated with finite precision.
            The ideal and quantized responses are compared sample by sample.

        </div>

        <div class="lc-card">

            <div class="lc-card-title">
                What to look for
            </div>

            Although |α| &lt; 1 makes the ideal response decay to zero,
            finite-precision quantization can trap the recursion in a
            non-zero fixed point or a periodic limit cycle.

        </div>

    </div>

</div>
""")


# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------

control_style = {'description_width': '110px'}


alpha_slider = FloatSlider(
    value=0.625,
    min=-0.95,
    max=0.95,
    step=0.125,
    description='Coefficient α:',
    continuous_update=True,
    readout_format='.3f',
    style=control_style,
    layout=Layout(width='300px')
)


K_slider = IntSlider(
    value=3,
    min=2,
    max=10,
    step=1,
    description='Bits K:',
    continuous_update=True,
    style=control_style,
    layout=Layout(width='260px')
)


y0_slider = FloatSlider(
    value=0.375,
    min=-0.95,
    max=0.95,
    step=0.0625,
    description='Initial y[0]:',
    continuous_update=True,
    readout_format='.4f',
    style=control_style,
    layout=Layout(width='300px')
)


samples_slider = IntSlider(
    value=24,
    min=12,
    max=60,
    step=1,
    description='Samples:',
    continuous_update=True,
    style=control_style,
    layout=Layout(width='260px')
)


mode_buttons = ToggleButtons(
    options=['Rounding', 'Truncation'],
    value='Rounding',
    description='Quantizer:',
    style={'description_width': '80px'},
    layout=Layout(width='350px')
)


controls_row_1 = HBox(
    [
        alpha_slider,
        K_slider,
        mode_buttons
    ],
    layout=Layout(
        width='950px',
        justify_content='space-between'
    )
)


controls_row_2 = HBox(
    [
        y0_slider,
        samples_slider
    ],
    layout=Layout(
        width='590px',
        justify_content='space-between'
    )
)


controls_box = VBox(
    [
        controls_row_1,
        controls_row_2
    ],
    layout=Layout(
        width='960px',
        border='1px solid #d3d9df',
        padding='7px',
        overflow='visible'
    )
)


# ------------------------------------------------------------
# Plot function
# ------------------------------------------------------------

def plot_limit_cycle(alpha, K, y0, samples, mode):

    Delta = 2.0**(-K)


    y_ideal, y_quantized = simulate_first_order(
        alpha,
        K,
        y0,
        samples,
        mode
    )


    period, cycle = detect_limit_cycle(
        y_quantized,
        K
    )


    dead_zone = (0.5 * Delta) / (1.0 - abs(alpha))


    # --------------------------------------------------------
    # Status interpretation
    # --------------------------------------------------------

    if period is None:

        status = 'No confirmed cycle'

        detail = 'No repeating pattern detected in the displayed interval.'


    elif period == 1 and np.isclose(cycle[0], 0.0):

        status = 'Converges to zero'

        detail = 'The quantized recursion reaches the zero state.'


    elif period == 1:

        status = 'Period-1 limit cycle'

        detail = f'Fixed quantized value: {cycle[0]:.6f}'


    else:

        status = f'Period-{period} limit cycle'

        cycle_text = ', '.join([f'{value:.6f}' for value in cycle])

        detail = f'Repeating sequence: [{cycle_text}]'


    # --------------------------------------------------------
    # Figure: compact 2 x 2 grid
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(11.2, 7.0)
    )


    ax1 = axes[0, 0]

    ax2 = axes[0, 1]

    ax3 = axes[1, 0]

    ax4 = axes[1, 1]


    # ========================================================
    # PANEL 1 — TIME RESPONSE
    # ========================================================

    n = np.arange(samples)


    if mode == 'Rounding':

        ax1.axhspan(
            -dead_zone,
            dead_zone,
            alpha=0.10,
            label='Dead zone'
        )


    ax1.plot(
        n,
        y_ideal,
        '--',
        linewidth=1.6,
        label='Ideal response'
    )


    markerline, stemlines, baseline = ax1.stem(
        n,
        y_quantized,
        basefmt=' '
    )


    plt.setp(
        stemlines,
        linewidth=1.4
    )


    plt.setp(
        markerline,
        markersize=4
    )


    ax1.axhline(
        0.0,
        linewidth=0.8
    )


    ax1.set_title(
        'Ideal vs Quantized Response',
        fontsize=11
    )


    ax1.set_xlabel(
        'Sample index n'
    )


    ax1.set_ylabel(
        'Amplitude'
    )


    ax1.grid(
        True,
        linestyle=':',
        alpha=0.4
    )


    ax1.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.17),
        ncol=2,
        frameon=False,
        fontsize=8
    )


    # ========================================================
    # PANEL 2 — QUANTIZED MAP
    # ========================================================

    x_values = np.linspace(
        -1.0,
        1.0 - Delta,
        1200
    )


    map_values = quantize(
        alpha * x_values,
        K,
        mode
    )


    ax2.plot(
        x_values,
        map_values,
        linewidth=1.8,
        label=r'$Q(\alpha y)$'
    )


    ax2.plot(
        x_values,
        x_values,
        '--',
        linewidth=1.1,
        label=r'$y[n]=y[n-1]$'
    )


    ax2.set_title(
        'Quantized State Map',
        fontsize=11
    )


    ax2.set_xlabel(
        r'$y[n-1]$'
    )


    ax2.set_ylabel(
        r'$y[n]$'
    )


    ax2.set_xlim(
        -1.0,
        1.0
    )


    ax2.set_ylim(
        -1.0,
        1.0
    )


    ax2.grid(
        True,
        linestyle=':',
        alpha=0.4
    )


    ax2.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.17),
        ncol=2,
        frameon=False,
        fontsize=8
    )


    # ========================================================
    # PANEL 3 — COBWEB / STATE ITERATION
    # ========================================================

    ax3.plot(
        x_values,
        map_values,
        linewidth=1.5,
        label='Quantized map'
    )


    ax3.plot(
        x_values,
        x_values,
        '--',
        linewidth=1.0,
        label='Identity'
    )


    current = y_quantized[0]


    for iteration in range(min(samples - 1, 20)):

        next_value = quantize(
            alpha * current,
            K,
            mode
        )


        ax3.plot(
            [current, current],
            [current, next_value],
            linewidth=1.0
        )


        ax3.plot(
            [current, next_value],
            [next_value, next_value],
            linewidth=1.0
        )


        ax3.plot(
            current,
            next_value,
            'o',
            markersize=3
        )


        current = next_value


    ax3.set_title(
        'State Iteration',
        fontsize=11
    )


    ax3.set_xlabel(
        r'$y[n-1]$'
    )


    ax3.set_ylabel(
        r'$y[n]$'
    )


    ax3.set_xlim(
        -1.0,
        1.0
    )


    ax3.set_ylim(
        -1.0,
        1.0
    )


    ax3.grid(
        True,
        linestyle=':',
        alpha=0.4
    )


    # ========================================================
    # PANEL 4 — STATUS / NUMERICAL DATA
    # ========================================================

    ax4.axis(
        'off'
    )


    stored_y0 = y_quantized[0]


    info_text = (
        f'Current experiment\n'
        f'────────────────────────────\n'
        f'α                  = {alpha:.4f}\n'
        f'K                  = {K}\n'
        f'Δ                  = {Delta:.6f}\n'
        f'Initial y[0]       = {stored_y0:.6f}\n'
        f'Quantizer          = {mode}\n\n'
        f'Observed behavior\n'
        f'────────────────────────────\n'
        f'{status}\n\n'
        f'{detail}\n\n'
        f'Rounding dead-zone bound\n'
        f'────────────────────────────\n'
        f'|y| ≤ {dead_zone:.6f}'
    )


    ax4.text(
        0.05,
        0.95,
        info_text,
        transform=ax4.transAxes,
        ha='left',
        va='top',
        fontsize=10,
        family='monospace',
        bbox=dict(
            boxstyle='round,pad=0.7',
            facecolor='white',
            alpha=0.95
        )
    )


    # --------------------------------------------------------
    # Figure title
    # --------------------------------------------------------

    fig.suptitle(
        f'Zero-Input Limit-Cycle Experiment — α = {alpha:.3f}, K = {K}, {mode}',
        fontsize=13
    )


    plt.subplots_adjust(
        left=0.08,
        right=0.97,
        top=0.91,
        bottom=0.12,
        wspace=0.28,
        hspace=0.38
    )


    plt.show()

    plt.close(fig)


# ------------------------------------------------------------
# Interactive object
# ------------------------------------------------------------

widget_plot = interactive(
    plot_limit_cycle,
    alpha=alpha_slider,
    K=K_slider,
    y0=y0_slider,
    samples=samples_slider,
    mode=mode_buttons
)


plot_output = widget_plot.children[-1]

plot_output.layout = Layout(
    width='auto',
    overflow='visible'
)


# ------------------------------------------------------------
# Final layout
# ------------------------------------------------------------

main_layout = VBox(
    [
        title_html,
        controls_box,
        plot_output
    ],
    layout=Layout(
        width='960px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(style_html)

display(main_layout)